# ToolIbV2 Colab Training API PoC

Notebook này dựng một FastAPI tạm thời trên Google Colab, public bằng Cloudflare Quick Tunnel và chạy tối đa một job Ultralytics YOLO detection tại một thời điểm.

Phạm vi PoC:

- Dataset được ZIP sẵn trên Google Drive, sau đó giải nén vào local disk của Colab.
- API dùng Bearer token lấy từ Colab Secret `TOOLIB_COLAB_API_TOKEN`.
- Client chỉ được chọn model/epochs/batch/imgsz; không được gửi filesystem path.
- Artifact `best.pt`, `best.onnx` và manifest được copy về Google Drive.
- Quick Tunnel chỉ dùng để test tạm thời, không phải production endpoint.

Chạy các cell từ trên xuống. Không chạy lại cell API trong lúc model đang train.

In [ ]:
# 1. Install dependencies and Cloudflare Tunnel
%pip install -q "fastapi>=0.115,<1" "uvicorn>=0.30,<1" "ultralytics>=8.3,<9" "requests>=2.32,<3" "pyyaml>=6,<7" "python-multipart>=0.0.20,<1"

import os
import platform
import subprocess
import urllib.request
from pathlib import Path

machine = platform.machine().lower()
if machine in {"x86_64", "amd64"}:
    cloudflared_asset = "cloudflared-linux-amd64"
elif machine in {"aarch64", "arm64"}:
    cloudflared_asset = "cloudflared-linux-arm64"
else:
    raise RuntimeError(f"Unsupported Colab architecture: {machine}")

CLOUDFLARED_BIN = Path("/content/cloudflared")
cloudflared_url = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/"
    + cloudflared_asset
)
cloudflared_ready = False
if CLOUDFLARED_BIN.is_file():
    existing_version = subprocess.run(
        [str(CLOUDFLARED_BIN), "--version"],
        capture_output=True,
        text=True,
        check=False,
    )
    cloudflared_ready = existing_version.returncode == 0
    if cloudflared_ready:
        print("Reusing existing cloudflared binary:", existing_version.stdout.strip())

if not cloudflared_ready:
    cloudflared_download = Path("/content/cloudflared.download")
    urllib.request.urlretrieve(cloudflared_url, cloudflared_download)
    cloudflared_download.chmod(0o755)
    os.replace(cloudflared_download, CLOUDFLARED_BIN)

subprocess.run([str(CLOUDFLARED_BIN), "--version"], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Select a GPU runtime before continuing.")

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive, read the API token, and configure safe dataset snapshots
import json
import shutil
import uuid
import zipfile
from pathlib import Path, PurePosixPath

import yaml
from google.colab import drive, userdata

drive.mount("/content/drive")

MANUAL_ARCHIVE_INPUT = "/content/drive/MyDrive/ToolIb_PoC/dataset.zip"  # @param {type:"string"}
MANUAL_ARCHIVE = Path(MANUAL_ARCHIVE_INPUT) if MANUAL_ARCHIVE_INPUT.strip() else None
WORK_ROOT = Path("/content/toolib_poc")
DATASET_CACHE_ROOT = WORK_ROOT / "datasets"
DATASET_UPLOAD_ROOT = WORK_ROOT / "uploads"
LOCAL_RUN_ROOT = WORK_ROOT / "runs"
DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/ToolIb_PoC/datasets")
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/ToolIb_PoC/artifacts")
ALLOWED_MODELS = {"yolo11n.pt", "yolo11s.pt"}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MAX_DATASET_ARCHIVE_BYTES = 20 * 1024**3
MAX_DATASET_EXTRACT_BYTES = 50 * 1024**3

API_TOKEN = userdata.get("TOOLIB_COLAB_API_TOKEN")
if not API_TOKEN or len(API_TOKEN) < 16:
    raise RuntimeError(
        "Create Colab Secret TOOLIB_COLAB_API_TOKEN with at least 16 characters, "
        "grant notebook access, then rerun this cell."
    )

for directory in (
    WORK_ROOT,
    DATASET_CACHE_ROOT,
    DATASET_UPLOAD_ROOT,
    LOCAL_RUN_ROOT,
    DRIVE_DATASET_ROOT,
    DRIVE_ARTIFACT_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)

def canonical_dataset_id(value: str) -> str:
    try:
        parsed = uuid.UUID(str(value))
    except (TypeError, ValueError, AttributeError) as exc:
        raise ValueError("dataset_id must be a valid UUID.") from exc
    return str(parsed)

def resolve_under(root: Path, *parts: str) -> Path:
    resolved_root = root.resolve()
    candidate = resolved_root.joinpath(*parts).resolve()
    try:
        candidate.relative_to(resolved_root)
    except ValueError as exc:
        raise RuntimeError(f"Path escapes the allowed root: {candidate}") from exc
    return candidate

def safe_extract_zip(archive_path: Path, destination: Path) -> None:
    if not archive_path.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {archive_path}")
    if archive_path.stat().st_size > MAX_DATASET_ARCHIVE_BYTES:
        raise RuntimeError("Dataset archive exceeds the configured size limit.")

    destination_root = destination.resolve()
    if destination_root.exists():
        shutil.rmtree(destination_root)
    destination_root.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path, "r") as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) > MAX_DATASET_EXTRACT_BYTES:
            raise RuntimeError("Extracted dataset would exceed the configured size limit.")

        for member in members:
            member_name = member.filename.replace("\\", "/")
            member_parts = PurePosixPath(member_name).parts
            unix_mode = (member.external_attr >> 16) & 0o170000
            if (
                not member_name
                or "\x00" in member_name
                or PurePosixPath(member_name).is_absolute()
                or ".." in member_parts
                or unix_mode == 0o120000
            ):
                raise RuntimeError(f"Unsafe path in dataset archive: {member.filename}")

            target = resolve_under(destination_root, *member_parts)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member, "r") as source, target.open("wb") as output:
                shutil.copyfileobj(source, output, length=1024 * 1024)

def find_dataset_yaml(root: Path) -> Path:
    matches = sorted(
        [*root.rglob("data.yaml"), *root.rglob("data.yml")],
        key=lambda item: (len(item.relative_to(root).parts), str(item)),
    )
    if not matches:
        raise FileNotFoundError("No data.yaml or data.yml was found in the dataset archive.")
    if len(matches) > 1:
        print("Warning: multiple dataset YAML files found; using:", matches[0])
    return matches[0]

def configured_dataset_root(config: dict, yaml_path: Path, extract_root: Path) -> Path:
    configured_root = Path(str(config.get("path", ".")))
    if not configured_root.is_absolute():
        configured_root = yaml_path.parent / configured_root
    configured_root = configured_root.resolve()
    try:
        configured_root.relative_to(extract_root.resolve())
    except ValueError as exc:
        raise RuntimeError(
            f"data.yaml path escapes the extracted dataset root: {configured_root}"
        ) from exc
    return configured_root

def resolve_split_paths(
    config: dict,
    yaml_path: Path,
    split: str,
    extract_root: Path,
) -> list[Path]:
    raw_value = config.get(split)
    if raw_value is None:
        return []

    values = raw_value if isinstance(raw_value, list) else [raw_value]
    dataset_root = configured_dataset_root(config, yaml_path, extract_root)
    resolved_paths = []
    for value in values:
        split_path = Path(str(value))
        if not split_path.is_absolute():
            split_path = dataset_root / split_path
        split_path = split_path.resolve()
        try:
            split_path.relative_to(extract_root.resolve())
        except ValueError as exc:
            raise RuntimeError(f"Dataset split path escapes the snapshot: {split_path}") from exc
        if not split_path.exists():
            raise FileNotFoundError(f"Dataset split path does not exist: {split_path}")
        resolved_paths.append(split_path)
    return resolved_paths

def count_images(paths: list[Path], extract_root: Path) -> int:
    count = 0
    for path in paths:
        if path.is_dir():
            count += sum(
                1 for item in path.rglob("*") if item.is_file() and item.suffix.lower() in IMAGE_SUFFIXES
            )
        elif path.is_file() and path.suffix.lower() == ".txt":
            for raw_line in path.read_text(encoding="utf-8").splitlines():
                if not raw_line.strip():
                    continue
                image_path = Path(raw_line.strip())
                if not image_path.is_absolute():
                    image_path = path.parent / image_path
                image_path = image_path.resolve()
                try:
                    image_path.relative_to(extract_root.resolve())
                except ValueError as exc:
                    raise RuntimeError(f"Image list entry escapes the snapshot: {image_path}") from exc
                if not image_path.is_file():
                    raise FileNotFoundError(f"Image list entry does not exist: {image_path}")
                count += 1
    return count

def prepare_runtime_dataset(dataset_id: str, archive_path: Path) -> dict:
    dataset_id = canonical_dataset_id(dataset_id)
    dataset_root = resolve_under(DATASET_CACHE_ROOT, dataset_id)
    extract_root = dataset_root / "dataset"
    safe_extract_zip(archive_path, extract_root)
    source_yaml = find_dataset_yaml(extract_root)

    with source_yaml.open("r", encoding="utf-8") as stream:
        config = yaml.safe_load(stream) or {}
    if not isinstance(config, dict):
        raise RuntimeError("data.yaml must contain a YAML mapping.")
    if "train" not in config or "val" not in config:
        raise RuntimeError("data.yaml must define both train and val splits.")
    if not config.get("names"):
        raise RuntimeError("data.yaml must define at least one class in names.")

    class_count = len(config["names"])
    configured_nc = config.get("nc")
    if configured_nc is not None and int(configured_nc) != class_count:
        raise RuntimeError(
            f"data.yaml nc={configured_nc} does not match names count={class_count}."
        )

    dataset_root_from_yaml = configured_dataset_root(config, source_yaml, extract_root)
    train_paths = resolve_split_paths(config, source_yaml, "train", extract_root)
    val_paths = resolve_split_paths(config, source_yaml, "val", extract_root)
    test_paths = resolve_split_paths(config, source_yaml, "test", extract_root)
    train_count = count_images(train_paths, extract_root)
    val_count = count_images(val_paths, extract_root)
    test_count = count_images(test_paths, extract_root)
    if train_count == 0 or val_count == 0:
        raise RuntimeError("The dataset snapshot must contain non-empty train and val splits.")

    runtime_config = dict(config)
    runtime_config["path"] = str(dataset_root_from_yaml)
    runtime_yaml = dataset_root / "runtime_data.yaml"
    with runtime_yaml.open("w", encoding="utf-8") as stream:
        yaml.safe_dump(
            runtime_config,
            stream,
            default_flow_style=False,
            sort_keys=False,
            allow_unicode=True,
        )

    return {
        "dataset_id": dataset_id,
        "runtime_yaml": str(runtime_yaml),
        "source_yaml": str(source_yaml),
        "train_count": train_count,
        "val_count": val_count,
        "test_count": test_count,
        "class_names": config["names"],
    }

DATASET_CACHE: dict[str, dict] = {}
DEFAULT_DATASET: dict | None = None

def resolve_training_dataset(dataset_id: str | None) -> dict:
    if dataset_id:
        canonical_id = canonical_dataset_id(dataset_id)
        cached = DATASET_CACHE.get(canonical_id)
        if cached and Path(cached["runtime_yaml"]).is_file():
            return cached
        drive_archive = resolve_under(DRIVE_DATASET_ROOT, canonical_id, "dataset.zip")
        prepared = prepare_runtime_dataset(canonical_id, drive_archive)
        DATASET_CACHE[canonical_id] = prepared
        return prepared
    if DEFAULT_DATASET is not None:
        return DEFAULT_DATASET
    raise FileNotFoundError(
        "No dataset_id was supplied and no manual fallback archive is available. "
        "Prepare and upload a dataset from ToolIbV2 first."
    )

if MANUAL_ARCHIVE is not None and MANUAL_ARCHIVE.is_file():
    manual_id = str(uuid.uuid5(uuid.NAMESPACE_URL, str(MANUAL_ARCHIVE.resolve())))
    DEFAULT_DATASET = prepare_runtime_dataset(manual_id, MANUAL_ARCHIVE)
    DATASET_CACHE[manual_id] = DEFAULT_DATASET
    print("Manual fallback dataset ready:", DEFAULT_DATASET["runtime_yaml"])
else:
    print("No manual dataset.zip found. Use ToolIbV2 to prepare and upload a snapshot.")

print("Drive dataset root:", DRIVE_DATASET_ROOT)
print("Drive artifact root:", DRIVE_ARTIFACT_ROOT)

In [ ]:
# 3. Define the authenticated FastAPI app and the single-GPU job worker
import copy
import hashlib
import json
import secrets
import shutil
import threading
import uuid
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

from fastapi import Depends, FastAPI, File, Form, HTTPException, Request, UploadFile, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from pydantic import BaseModel, ConfigDict, Field, field_validator
from ultralytics import YOLO

app = FastAPI(title="ToolIbV2 Colab Training Worker", version="0.5.0")
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5000",
        "https://localhost:5000",
        "http://127.0.0.1:5000",
        "https://127.0.0.1:5000",
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type", "Idempotency-Key"],
)
bearer_scheme = HTTPBearer(auto_error=False)
JOB_LOCK = threading.Lock()
JOBS: dict[str, dict] = {}
IDEMPOTENCY_INDEX: dict[str, str] = {}
ACTIVE_JOB_ID: str | None = None
TRAIN_EXECUTOR = ThreadPoolExecutor(max_workers=1, thread_name_prefix="toolib-yolo")
ACTIVE_STATES = {"queued", "running"}

class TrainRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    model: str = "yolo11n.pt"
    epochs: int = Field(default=1, ge=1, le=100)
    batch: int = Field(default=4, ge=1, le=32)
    imgsz: Literal[320, 416, 512, 640, 768] = 640
    dataset_id: str | None = None
    idempotency_key: str | None = Field(default=None, min_length=1, max_length=100)

    @field_validator("model")
    @classmethod
    def validate_model(cls, value: str) -> str:
        if value not in ALLOWED_MODELS:
            raise ValueError(f"model must be one of {sorted(ALLOWED_MODELS)}")
        return value

    @field_validator("dataset_id")
    @classmethod
    def validate_dataset_id(cls, value: str | None) -> str | None:
        return canonical_dataset_id(value) if value else None

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def require_api_token(
    credentials: HTTPAuthorizationCredentials | None = Depends(bearer_scheme),
) -> None:
    if (
        credentials is None
        or credentials.scheme.lower() != "bearer"
        or not secrets.compare_digest(credentials.credentials, API_TOKEN)
    ):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or missing API token.",
            headers={"WWW-Authenticate": "Bearer"},
        )

def get_public_job(job_id: str) -> dict:
    with JOB_LOCK:
        job = JOBS.get(job_id)
        if job is None:
            raise HTTPException(status_code=404, detail="Job not found.")
        return copy.deepcopy(job)

def update_job(job_id: str, **updates) -> None:
    with JOB_LOCK:
        if job_id in JOBS:
            JOBS[job_id].update(updates)

def restore_persisted_jobs() -> int:
    restored = 0
    if not DRIVE_ARTIFACT_ROOT.is_dir():
        return restored

    for artifact_dir in DRIVE_ARTIFACT_ROOT.iterdir():
        if not artifact_dir.is_dir():
            continue
        try:
            job_id = str(uuid.UUID(artifact_dir.name))
        except ValueError:
            continue

        manifest_path = artifact_dir / "manifest.json"
        failure_path = artifact_dir / "failure.json"
        source_path = manifest_path if manifest_path.is_file() else failure_path
        if not source_path.is_file():
            continue
        try:
            persisted = json.loads(source_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if str(persisted.get("job_id") or "") != job_id:
            continue

        request_payload = dict(persisted.get("request") or {})
        model = persisted.get("model") or request_payload.get("model")
        epochs = int(persisted.get("epochs") or request_payload.get("epochs") or 1)
        batch = int(persisted.get("batch") or request_payload.get("batch") or 1)
        imgsz = int(persisted.get("imgsz") or request_payload.get("imgsz") or 640)
        dataset_id = persisted.get("dataset_id") or request_payload.get("dataset_id")
        idempotency_key = persisted.get("idempotency_key") or request_payload.pop("idempotency_key", None)
        status_value = str(persisted.get("status") or "failed")
        artifacts = {}
        if (artifact_dir / "best.pt").is_file():
            artifacts["pt"] = str(artifact_dir / "best.pt")
        if (artifact_dir / "best.onnx").is_file():
            artifacts["onnx"] = str(artifact_dir / "best.onnx")
        if manifest_path.is_file():
            artifacts["manifest"] = str(manifest_path)
        if failure_path.is_file():
            artifacts["failure"] = str(failure_path)

        job = {
            "job_id": job_id,
            "status": status_value,
            "created_at": persisted.get("created_at") or persisted.get("finished_at"),
            "started_at": persisted.get("started_at"),
            "finished_at": persisted.get("finished_at"),
            "current_epoch": epochs if status_value == "succeeded" else 0,
            "total_epochs": epochs,
            "message": (
                "Training and ONNX export completed."
                if status_value == "succeeded"
                else "Training job failed."
            ),
            "error": persisted.get("error"),
            "artifacts": artifacts or None,
            "dataset_id": dataset_id,
            "idempotency_key": idempotency_key,
            "request": {
                "model": model,
                "epochs": epochs,
                "batch": batch,
                "imgsz": imgsz,
                "dataset_id": dataset_id,
            },
        }
        with JOB_LOCK:
            JOBS[job_id] = job
            if idempotency_key:
                IDEMPOTENCY_INDEX[str(idempotency_key)] = job_id
        restored += 1
    return restored

RESTORED_JOB_COUNT = restore_persisted_jobs()

def normalize_export_path(export_result) -> Path:
    value = export_result
    if isinstance(value, (list, tuple)):
        if not value:
            raise RuntimeError("Ultralytics export returned no artifact path.")
        value = value[0]
    path = Path(str(value)).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Exported ONNX file was not found: {path}")
    return path

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

def sha256_path(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

@app.post("/api/datasets", dependencies=[Depends(require_api_token)])
async def upload_dataset_snapshot(
    dataset_id: str = Form(...),
    sha256: str = Form(...),
    archive: UploadFile = File(...),
):
    try:
        canonical_id = canonical_dataset_id(dataset_id)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    expected_sha256 = sha256.strip().lower()
    try:
        if len(expected_sha256) != 64:
            raise ValueError
        int(expected_sha256, 16)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail="sha256 must contain 64 hexadecimal characters.") from exc

    drive_dataset_dir = resolve_under(DRIVE_DATASET_ROOT, canonical_id)
    drive_archive = drive_dataset_dir / "dataset.zip"
    drive_manifest = drive_dataset_dir / "manifest.json"
    existing_manifest = None
    if drive_manifest.is_file():
        try:
            existing_manifest = json.loads(drive_manifest.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as exc:
            raise HTTPException(
                status_code=status.HTTP_409_CONFLICT,
                detail="dataset_id has an unreadable Drive manifest.",
            ) from exc
        if not isinstance(existing_manifest, dict):
            raise HTTPException(
                status_code=status.HTTP_409_CONFLICT,
                detail="dataset_id has an invalid Drive manifest.",
            )
        existing_sha256 = existing_manifest.get("sha256")
        if existing_sha256 and existing_sha256 != expected_sha256:
            raise HTTPException(
                status_code=status.HTTP_409_CONFLICT,
                detail="dataset_id already exists with a different checksum.",
            )
    if (
        drive_archive.is_file()
        and existing_manifest is not None
        and sha256_path(drive_archive) == expected_sha256
    ):
        return {
            "status": "existing",
            "dataset_id": canonical_id,
            "sha256": expected_sha256,
            "drive_path": str(drive_archive),
            "stats": existing_manifest.get("stats", {}),
        }

    upload_path = resolve_under(DATASET_UPLOAD_ROOT, f"{canonical_id}.upload")
    if upload_path.exists():
        upload_path.unlink()

    digest = hashlib.sha256()
    uploaded_bytes = 0
    try:
        with upload_path.open("wb") as output:
            while True:
                chunk = await archive.read(1024 * 1024)
                if not chunk:
                    break
                uploaded_bytes += len(chunk)
                if uploaded_bytes > MAX_DATASET_ARCHIVE_BYTES:
                    raise HTTPException(status_code=413, detail="Dataset archive exceeds the size limit.")
                digest.update(chunk)
                output.write(chunk)

        actual_sha256 = digest.hexdigest()
        if actual_sha256 != expected_sha256:
            raise HTTPException(status_code=422, detail="Dataset archive checksum does not match sha256.")

        prepared = prepare_runtime_dataset(canonical_id, upload_path)
        if drive_archive.is_file() and sha256_path(drive_archive) != expected_sha256:
            raise HTTPException(
                status_code=status.HTTP_409_CONFLICT,
                detail="dataset_id already exists with a different archive.",
            )

        drive_dataset_dir.mkdir(parents=True, exist_ok=True)
        staged_archive = drive_dataset_dir / ".dataset.zip.upload"
        shutil.copy2(upload_path, staged_archive)
        staged_archive.replace(drive_archive)
        stats = {
            "train": prepared["train_count"],
            "val": prepared["val_count"],
            "test": prepared["test_count"],
            "classes": prepared["class_names"],
        }
        manifest = {
            "dataset_id": canonical_id,
            "sha256": expected_sha256,
            "archive_size": uploaded_bytes,
            "drive_path": str(drive_archive),
            "stats": stats,
            "uploaded_at": utc_now(),
        }
        write_json(drive_manifest, manifest)
        DATASET_CACHE[canonical_id] = prepared
        return {
            "status": "uploaded",
            "dataset_id": canonical_id,
            "sha256": expected_sha256,
            "drive_path": str(drive_archive),
            "stats": stats,
        }
    except HTTPException:
        raise
    except (OSError, RuntimeError, ValueError, zipfile.BadZipFile, yaml.YAMLError) as exc:
        raise HTTPException(status_code=400, detail=f"Dataset snapshot rejected: {exc}") from exc
    finally:
        await archive.close()
        if upload_path.exists():
            upload_path.unlink()

def run_training_job(job_id: str, request_data: TrainRequest, dataset_info: dict) -> None:
    global ACTIVE_JOB_ID

    artifact_dir = DRIVE_ARTIFACT_ROOT / job_id
    try:
        artifact_dir.mkdir(parents=True, exist_ok=False)
        update_job(
            job_id,
            status="running",
            started_at=utc_now(),
            message=f"Loading dataset {dataset_info['dataset_id'][:8]} and starting training.",
        )

        model = YOLO(request_data.model)

        def on_train_epoch_end(trainer) -> None:
            current_epoch = int(getattr(trainer, "epoch", 0)) + 1
            total_epochs = int(getattr(trainer, "epochs", request_data.epochs))
            update_job(
                job_id,
                current_epoch=current_epoch,
                total_epochs=total_epochs,
                message=f"Training epoch {current_epoch}/{total_epochs}.",
            )

        model.add_callback("on_train_epoch_end", on_train_epoch_end)
        model.train(
            data=dataset_info["runtime_yaml"],
            epochs=request_data.epochs,
            batch=request_data.batch,
            imgsz=request_data.imgsz,
            device=0,
            workers=2,
            project=str(LOCAL_RUN_ROOT),
            name=job_id,
            exist_ok=False,
        )

        trainer = getattr(model, "trainer", None)
        save_dir_value = getattr(trainer, "save_dir", None)
        if save_dir_value is None:
            raise RuntimeError("Ultralytics did not expose the training save directory.")

        save_dir = Path(str(save_dir_value)).resolve()
        best_pt = save_dir / "weights" / "best.pt"
        if not best_pt.is_file():
            raise FileNotFoundError(f"Training completed but best.pt was not found: {best_pt}")

        update_job(job_id, message="Exporting best.pt to ONNX.")
        export_model = YOLO(str(best_pt))
        export_result = export_model.export(
            format="onnx",
            dynamic=True,
            imgsz=request_data.imgsz,
        )
        exported_onnx = normalize_export_path(export_result)

        pt_target = artifact_dir / "best.pt"
        onnx_target = artifact_dir / "best.onnx"
        shutil.copy2(best_pt, pt_target)
        shutil.copy2(exported_onnx, onnx_target)

        copied_training_files = []
        for filename in ("results.csv", "args.yaml"):
            source = save_dir / filename
            if source.is_file():
                destination = artifact_dir / filename
                shutil.copy2(source, destination)
                copied_training_files.append(str(destination))

        manifest_path = artifact_dir / "manifest.json"
        job_snapshot = get_public_job(job_id)
        manifest = {
            "job_id": job_id,
            "status": "succeeded",
            "created_at": job_snapshot.get("created_at"),
            "started_at": job_snapshot.get("started_at"),
            "model": request_data.model,
            "epochs": request_data.epochs,
            "batch": request_data.batch,
            "imgsz": request_data.imgsz,
            "dataset_id": dataset_info["dataset_id"],
            "idempotency_key": job_snapshot.get("idempotency_key"),
            "request": job_snapshot.get("request"),
            "dataset_yaml": dataset_info["runtime_yaml"],
            "training_save_dir": str(save_dir),
            "artifacts": {
                "pt": str(pt_target),
                "onnx": str(onnx_target),
                "training_files": copied_training_files,
            },
            "finished_at": utc_now(),
        }
        write_json(manifest_path, manifest)

        update_job(
            job_id,
            status="succeeded",
            finished_at=manifest["finished_at"],
            current_epoch=request_data.epochs,
            total_epochs=request_data.epochs,
            message="Training and ONNX export completed.",
            error=None,
            artifacts={
                "pt": str(pt_target),
                "onnx": str(onnx_target),
                "manifest": str(manifest_path),
            },
        )
    except Exception as exc:
        error_message = f"{type(exc).__name__}: {exc}"
        failure_path = artifact_dir / "failure.json"
        failure = {
            "job_id": job_id,
            "status": "failed",
            "request": request_data.model_dump(),
            "error": error_message,
            "finished_at": utc_now(),
        }
        try:
            write_json(failure_path, failure)
        except Exception as persist_exc:
            error_message += f"; could not persist failure: {persist_exc}"

        update_job(
            job_id,
            status="failed",
            finished_at=failure["finished_at"],
            message="Training job failed.",
            error=error_message,
            artifacts={"failure": str(failure_path)} if failure_path.exists() else None,
        )
    finally:
        with JOB_LOCK:
            if ACTIVE_JOB_ID == job_id:
                ACTIVE_JOB_ID = None

@app.get("/health")
def health() -> dict:
    with JOB_LOCK:
        active_job_id = ACTIVE_JOB_ID
    gpu_available = bool(torch.cuda.is_available())
    return {
        "status": "online",
        "gpu_available": gpu_available,
        "gpu_name": torch.cuda.get_device_name(0) if gpu_available else None,
        "active_job_id": active_job_id,
        "idempotent_submit": True,
        "capabilities": {
            "dataset_upload": True,
            "training": True,
            "artifact_download": ["onnx"],
            "max_concurrent_jobs": 1,
            "idempotent_submit": True,
        },
    }

@app.post("/api/train", dependencies=[Depends(require_api_token)])
def start_train(request_data: TrainRequest, request: Request):
    global ACTIVE_JOB_ID
    header_key = str(request.headers.get("Idempotency-Key") or "").strip() or None
    body_key = request_data.idempotency_key
    if header_key and body_key and header_key != body_key:
        raise HTTPException(status_code=400, detail="Idempotency key mismatch.")
    idempotency_key = header_key or body_key

    try:
        dataset_info = resolve_training_dataset(request_data.dataset_id)
    except (OSError, RuntimeError, ValueError, zipfile.BadZipFile, yaml.YAMLError) as exc:
        raise HTTPException(status_code=400, detail=f"Dataset is not ready: {exc}") from exc

    with JOB_LOCK:
        if idempotency_key and idempotency_key in IDEMPOTENCY_INDEX:
            replay_job_id = IDEMPOTENCY_INDEX[idempotency_key]
            replay_job = copy.deepcopy(JOBS[replay_job_id])
            return JSONResponse(
                status_code=status.HTTP_200_OK,
                content={
                    "job_id": replay_job_id,
                    "status": replay_job["status"],
                    "dataset_id": replay_job["dataset_id"],
                    "idempotent_replay": True,
                },
            )

        if ACTIVE_JOB_ID is not None:
            active_job = JOBS.get(ACTIVE_JOB_ID, {})
            if active_job.get("status") in ACTIVE_STATES:
                raise HTTPException(
                    status_code=status.HTTP_409_CONFLICT,
                    detail={
                        "message": "A training job is already active.",
                        "active_job_id": ACTIVE_JOB_ID,
                    },
                )

        job_id = str(uuid.uuid4())
        ACTIVE_JOB_ID = job_id
        JOBS[job_id] = {
            "job_id": job_id,
            "status": "queued",
            "created_at": utc_now(),
            "started_at": None,
            "finished_at": None,
            "current_epoch": 0,
            "total_epochs": request_data.epochs,
            "message": "Training job accepted.",
            "error": None,
            "artifacts": None,
            "dataset_id": dataset_info["dataset_id"],
            "idempotency_key": idempotency_key,
            "request": request_data.model_dump(exclude={"idempotency_key"}),
        }
        if idempotency_key:
            IDEMPOTENCY_INDEX[idempotency_key] = job_id

    try:
        TRAIN_EXECUTOR.submit(run_training_job, job_id, request_data, dataset_info)
    except Exception as exc:
        with JOB_LOCK:
            ACTIVE_JOB_ID = None
            JOBS[job_id].update(
                status="failed",
                finished_at=utc_now(),
                message="Could not submit training job.",
                error=f"{type(exc).__name__}: {exc}",
            )
        raise HTTPException(status_code=500, detail="Could not submit training job.") from exc

    return JSONResponse(
        status_code=status.HTTP_202_ACCEPTED,
        content={
            "job_id": job_id,
            "status": "queued",
            "dataset_id": dataset_info["dataset_id"],
            "idempotent_replay": False,
        },
    )

@app.get("/api/jobs/{job_id}", dependencies=[Depends(require_api_token)])
def get_job(job_id: str) -> dict:
    return get_public_job(job_id)

@app.get(
    "/api/jobs/{job_id}/artifacts/onnx",
    dependencies=[Depends(require_api_token)],
)
def download_job_onnx(job_id: str):
    job = get_public_job(job_id)
    if job.get("status") != "succeeded":
        raise HTTPException(status_code=409, detail="Training job has not succeeded.")

    onnx_value = (job.get("artifacts") or {}).get("onnx")
    if not onnx_value:
        raise HTTPException(status_code=404, detail="ONNX artifact is not available.")

    artifact_root = (DRIVE_ARTIFACT_ROOT / job_id).resolve()
    onnx_path = Path(str(onnx_value)).resolve()
    try:
        onnx_path.relative_to(artifact_root)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail="Invalid ONNX artifact path.") from exc
    if onnx_path.suffix.lower() != ".onnx" or not onnx_path.is_file():
        raise HTTPException(status_code=404, detail="ONNX artifact file was not found.")

    return FileResponse(
        path=onnx_path,
        media_type="application/octet-stream",
        filename=f"{job_id[:8]}-best.onnx",
    )

print("FastAPI application created.")
print("Allowed models:", sorted(ALLOWED_MODELS))
print("Restored persisted jobs from Drive:", RESTORED_JOB_COUNT)

In [ ]:
# 4. Start Uvicorn, wait for health, then create the temporary Quick Tunnel URL
import queue
import re
import subprocess
import threading
import time

import requests
import uvicorn

LOCAL_API_URL = "http://127.0.0.1:8000"

old_tunnel = globals().get("TUNNEL_PROCESS")
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()

old_server = globals().get("UVICORN_SERVER")
old_server_thread = globals().get("UVICORN_THREAD")
if old_server is not None:
    old_server.should_exit = True
if old_server_thread is not None and old_server_thread.is_alive():
    old_server_thread.join(timeout=5)

class NotebookUvicornServer(uvicorn.Server):
    def install_signal_handlers(self) -> None:
        return None

uvicorn_config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000,
    log_level="info",
    access_log=True,
)
UVICORN_SERVER = NotebookUvicornServer(uvicorn_config)
UVICORN_THREAD = threading.Thread(target=UVICORN_SERVER.run, daemon=True)
UVICORN_THREAD.start()

health_deadline = time.time() + 30
while time.time() < health_deadline:
    try:
        response = requests.get(f"{LOCAL_API_URL}/health", timeout=2)
        if response.status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    raise RuntimeError("Uvicorn did not become healthy within 30 seconds.")

TUNNEL_PROCESS = subprocess.Popen(
    [
        str(CLOUDFLARED_BIN),
        "tunnel",
        "--url",
        LOCAL_API_URL,
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_lines = queue.Queue()

def collect_tunnel_output(stream, output_queue) -> None:
    for line in iter(stream.readline, ""):
        output_queue.put(line)

TUNNEL_LOG_THREAD = threading.Thread(
    target=collect_tunnel_output,
    args=(TUNNEL_PROCESS.stdout, tunnel_lines),
    daemon=True,
)
TUNNEL_LOG_THREAD.start()

PUBLIC_API_URL = None
tunnel_deadline = time.time() + 45
url_pattern = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")

while time.time() < tunnel_deadline:
    if TUNNEL_PROCESS.poll() is not None:
        raise RuntimeError(f"cloudflared exited with code {TUNNEL_PROCESS.returncode}")
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        continue
    match = url_pattern.search(line)
    if match:
        PUBLIC_API_URL = match.group(0)
        break

if PUBLIC_API_URL is None:
    TUNNEL_PROCESS.terminate()
    raise RuntimeError("Quick Tunnel URL was not reported within 45 seconds.")

print("Local API:", LOCAL_API_URL)
print("Temporary public API:", PUBLIC_API_URL)
print("Health endpoint:", f"{PUBLIC_API_URL}/health")
print("The bearer token remains in Colab Secret and was not printed.")

In [ ]:
# 5. Optional one-epoch end-to-end smoke test
RUN_TRAIN_SMOKE = False  # @param {type:"boolean"}
SMOKE_MODEL = "yolo11n.pt"  # @param ["yolo11n.pt", "yolo11s.pt"]
SMOKE_EPOCHS = 1  # @param {type:"integer"}
SMOKE_BATCH = 4  # @param {type:"integer"}
SMOKE_IMGSZ = 640  # @param [320, 416, 512, 640, 768] {type:"raw"}
SMOKE_DATASET_ID = ""  # @param {type:"string"}

import json
import time

import requests

if not RUN_TRAIN_SMOKE:
    print("Smoke training is disabled. Set RUN_TRAIN_SMOKE=True when ready.")
else:
    auth_headers = {"Authorization": f"Bearer {API_TOKEN}"}
    payload = {
        "model": SMOKE_MODEL,
        "epochs": SMOKE_EPOCHS,
        "batch": SMOKE_BATCH,
        "imgsz": SMOKE_IMGSZ,
    }
    if SMOKE_DATASET_ID.strip():
        payload["dataset_id"] = SMOKE_DATASET_ID.strip()
    submit_response = requests.post(
        f"{PUBLIC_API_URL}/api/train",
        headers=auth_headers,
        json=payload,
        timeout=30,
    )
    print("Submit status:", submit_response.status_code)
    print(json.dumps(submit_response.json(), ensure_ascii=False, indent=2))
    submit_response.raise_for_status()
    submitted_job_id = submit_response.json()["job_id"]

    while True:
        status_response = requests.get(
            f"{PUBLIC_API_URL}/api/jobs/{submitted_job_id}",
            headers=auth_headers,
            timeout=30,
        )
        status_response.raise_for_status()
        job = status_response.json()
        print(
            job["status"],
            job.get("message"),
            f"epoch={job.get('current_epoch')}/{job.get('total_epochs')}",
        )
        if job["status"] in {"succeeded", "failed"}:
            print(json.dumps(job, ensure_ascii=False, indent=2))
            break
        time.sleep(10)

## Hoàn tất và cleanup

Khi job thành công, lấy `best.onnx` từ `MyDrive/ToolIb_PoC/artifacts/<job_id>/` và import vào ToolIbV2.

Khi test xong:

```python
if TUNNEL_PROCESS.poll() is None:
    TUNNEL_PROCESS.terminate()
UVICORN_SERVER.should_exit = True
TRAIN_EXECUTOR.shutdown(wait=False, cancel_futures=True)
```

Disconnect Colab runtime để giải phóng GPU. URL Quick Tunnel sẽ không còn sử dụng được sau khi tunnel hoặc runtime dừng.